# <font color='white'> Lab-06 (Solution) <font>

## <font color='blue'> Example of Wordnet <font>
    

In [68]:
import nltk
from nltk.corpus import wordnet as wn

# Example word
word = "hate"

# Get synsets (different meanings)
synsets = wn.synsets(word)

print(f"\nWord: \033[1;34m{word}\033[0m")  # Blue bold

if not synsets:
    print("No synsets found.")
else:
    print(f"\033[1;32m Found {len(synsets)} synsets:\033[0m\n")
    for i, syn in enumerate(synsets, 1):
        print(f"Synset {i}: \033[1;33m{syn.name()}\033[0m")  # Yellow
        print(f"  Definition: {syn.definition()}")
        examples = syn.examples()
        if examples:
            print(f"  Examples: {examples}")
        else:
            print(f"  Examples: None")
        print("-" * 50)


Word: hate
 Found 2 synsets:

Synset 1: hate.n.01
  Definition: the emotion of intense dislike; a feeling of dislike so strong that it demands action
  Examples: None
--------------------------------------------------
Synset 2: hate.v.01
  Definition: dislike intensely; feel antipathy or aversion towards
  Examples: ['I hate Mexican food', 'She detests politicians']
--------------------------------------------------


## <font color='blue'> Data Preparation <font>
    
Download Emotios dataset from Kaggle https://www.kaggle.com/datasets/nelgiriyewithana/emotions
    
Use Pandas to load the dataset and display the first fifteen rows of the dataset.

In [69]:
# (0.1)
import pandas as pd
df = pd.read_csv('text.csv',nrows=2000)  #subset = df.iloc[1000:3000] 
df.info()
print(df['label'].value_counts())  #to see percentage use value_counts(normalize=True)
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  2000 non-null   int64 
 1   text        2000 non-null   object
 2   label       2000 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 47.0+ KB
label
1    695
0    583
3    269
4    225
2    157
5     71
Name: count, dtype: int64


,Unnamed: 0,text,label
0,0,i just feel really helpless and heavy hearted,4
1,1,ive enjoyed being able to slouch about relax a...,0
2,2,i gave up my internship with the dmrg and am f...,4
3,3,i dont know i feel so lost,0
4,4,i am a kindergarten teacher and i am thoroughl...,4


## <font color='blue'> Problem 1: Data Pre-processing <font>
    
   
(1.1) Write a function, data_preprocess, to tokenize text, remove stopwords and non-alphabetic tokens, apply lemmatization, and rejoin tokens into a single string.
    
(1.2) Load the dataset and apply the data_preprocess function to pre-process the text data. Preview the data to ensure pre-processing is as expected.
    

In [70]:
# (1.1)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download the required NLTK data
#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')

# Initialize the lemmatizer and stopwords list
lem = WordNetLemmatizer()
stop_words = stopwords.words('english')

def data_preprocess(text):
    # Tokenize text
    wtokens = word_tokenize(text)

   # Filtering tokens
    t_filtered = []
    for t in wtokens:
        # Convert token to lowercase and check if it's not in stopwords and is alphabetic
        if t.lower() not in stop_words and t.isalpha():
        # Add the lowercase token to filtered_tokens
            t_filtered.append(t.lower())

# Lemmatization
    t_lemmatized = []
    for t in t_filtered:
    # Lemmatize token
        lemma_t = lem.lemmatize(t)
    # Add lemmatized token to lemmatized_tokens
        t_lemmatized.append(lemma_t)

    # Rejoin the processed tokens into a single string
    return " ".join(t_lemmatized) 

# (1.2)
df['cleaned_text'] = df['text'].apply(data_preprocess)
df.head(15)

,Unnamed: 0,text,label,cleaned_text
0,0,i just feel really helpless and heavy hearted,4,feel really helpless heavy hearted
1,1,ive enjoyed being able to slouch about relax a...,0,ive enjoyed able slouch relax unwind frankly n...
2,2,i gave up my internship with the dmrg and am f...,4,gave internship dmrg feeling distraught
3,3,i dont know i feel so lost,0,dont know feel lost
4,4,i am a kindergarten teacher and i am thoroughl...,4,kindergarten teacher thoroughly weary job take...
5,5,i was beginning to feel quite disheartened,0,beginning feel quite disheartened
6,6,i would think that whomever would be lucky eno...,2,would think whomever would lucky enough stay s...
7,7,i fear that they won t ever feel that deliciou...,1,fear ever feel delicious excitement christmas ...
8,8,im forever taking some time out to have a lie ...,5,im forever taking time lie feel weird
9,9,i can still lose the weight without feeling de...,0,still lose weight without feeling deprived


### <font color='blue'> Problem 2:  Sentiment analysis with TextBlob <font>

Use TextBlob to perform sentiment analysis on a sample of texts from the dataset.
    
Calculate the sentiment polarity and subjectivity of random 5 texts from the dataset and determine if each text is positive, negative, or neutral based on the polarity score.

In [71]:
from textblob import TextBlob
from prettytable import PrettyTable
import textwrap

# Sample 5 random texts
random_text = df['cleaned_text'].sample(5).values


# Create PrettyTable
table = PrettyTable()
table.field_names = ["#", "Text", "Polarity", "Subjectivity", "Sentiment"]

# Set maximum width for text column
MAX_WIDTH = 50

# Analyze sentiment and add rows
for i, text in enumerate(random_text, start=1):
    tb = TextBlob(text)
    score = tb.sentiment.polarity
    subj = tb.sentiment.subjectivity
   # print(f"({i}) {text}\nSentiment polarity: {round(score,2)}\nSentiment subjectivity: {round(subj,2)}\n")

    # Determine sentiment category
    if score > 0:
        sentiment = "Positive"
    elif score < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"
    
    # Wrap long text to multiple lines
    wrapped_text = "\n".join(textwrap.wrap(text, width=MAX_WIDTH))
    
    table.add_row([i, wrapped_text, round(score, 2), round(subj, 2), sentiment])

# Print the table
print(table)

+---+----------------------------------------------------+----------+--------------+-----------+
| # |                        Text                        | Polarity | Subjectivity | Sentiment |
+---+----------------------------------------------------+----------+--------------+-----------+
| 1 |                  feeling romantic                  |   0.0    |     0.5      |  Neutral  |
| 2 |   feel like boring as sell listen london calling   |   -1.0   |     1.0      |  Negative |
|   |                  still feel spark                  |          |              |           |
| 3 | feel foolish hoping anything life think wrong side |   0.15   |     0.65     |  Positive |
|   |                win lose hope faith                 |          |              |           |
| 4 |           also feel like way said funny            |   0.25   |     1.0      |  Positive |
| 5 |  last day school traditionally celebrated singing  |   0.17   |     0.41     |  Positive |
|   |             song student

## <font color='blue'> Problem 3:  Sentiment analysis with VADER <font>
    
Analyze the sentiment of same 5 random texts using VADER.
    
Discuss how VADER’s scores (positive, negative, neutral, compound) provide a nuanced view of sentiment.

In [72]:
#!pip install vader

In [73]:
#nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vdr = SentimentIntensityAnalyzer()
    
    # Create PrettyTable
table = PrettyTable()
table.field_names = ["#", "Text", "Positive", "Neutral", "Negative", "Compound","Sentiment"]

MAX_WIDTH = 50  # wrap long text
for i, text in enumerate(random_text,start=1): # Analyze sentiment and add rows
    vsa = vdr.polarity_scores(text)
    #print(f"({i}) {text}\nVADER Scores: {vsa}\n")
    wrapped_text = "\n".join(textwrap.wrap(text, width=MAX_WIDTH))
    # Determine sentiment category
    compound = vsa['compound']
    if compound >= 0.05:
        sentiment = "Positive"
    elif compound <= -0.05:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"
    
    # Wrap long text
    wrapped_text = "\n".join(textwrap.wrap(text, width=MAX_WIDTH))
    
    table.add_row([i,wrapped_text,
        round(vsa['pos'], 2),
        round(vsa['neu'], 2),
        round(vsa['neg'], 2),
        round(compound, 2),
        sentiment
    ])

print(table)

+---+----------------------------------------------------+----------+---------+----------+----------+-----------+
| # |                        Text                        | Positive | Neutral | Negative | Compound | Sentiment |
+---+----------------------------------------------------+----------+---------+----------+----------+-----------+
| 1 |                  feeling romantic                  |   1.0    |   0.0   |   0.0    |   0.49   |  Positive |
| 2 |   feel like boring as sell listen london calling   |   0.3    |   0.54  |   0.16   |   0.27   |  Positive |
|   |                  still feel spark                  |          |         |          |          |           |
| 3 | feel foolish hoping anything life think wrong side |   0.49   |   0.2   |   0.31   |   0.66   |  Positive |
|   |                win lose hope faith                 |          |         |          |          |           |
| 4 |           also feel like way said funny            |   0.57   |   0.43  |   0.0   

## <font color='blue'> Problem 4:  Sentiment analysis Using AFINN <font>
        
(4.1) Install the 'afinn' module.
    
(4.2) Use the AFINN library to calculate sentiment scores for the same 5 random texts.
    
(4.3) Compare the scores against those obtained from TextBlob and VADER.

In [58]:
   
from afinn import Afinn

afinn = Afinn()
# Create PrettyTable
table = PrettyTable()
table.field_names = ["#", "Text", "AFINN Score", "Sentiment"]

MAX_WIDTH = 50  

for i, text in enumerate(random_text, start=1): # Analyze sentiment and add rows
    score = afinn.score(text)
    #print(f"({i}) {text}\nAFINN Score: {score}\n")

    # Determine sentiment category
    if score > 0:
        sentiment = "Positive"
    elif score < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"
    
    # Wrap long text
    wrapped_text = "\n".join(textwrap.wrap(text, width=MAX_WIDTH))
    
    table.add_row([i, wrapped_text, round(score, 2), sentiment])

# Print the table
print(table)


+---+----------------------------------------------------+-------------+-----------+
| # |                        Text                        | AFINN Score | Sentiment |
+---+----------------------------------------------------+-------------+-----------+
| 1 |    feel like teenager laughed looking williams     |     5.0     |  Positive |
|   |                  playful grin lip                  |             |           |
| 2 | hurry feel irritated understand mean bless family  |     -1.0    |  Negative |
|   |                keeping house order                 |             |           |
| 3 |     feel feel im stubborn nothing going change     |     -2.0    |  Negative |
| 4 | always needed always absolute best make feel vital |     9.0     |  Positive |
|   |  life always tried support life always tried best  |             |           |
|   |                       friend                       |             |           |
| 5 |  read feel like im something worthwhile platform   |     2.

## <font color='blue'> Problem 5:  Build a Simple Classifier for Sentiment Analysis with scikit-learn <font>

(5.1) Preprocess the dataset to create a binary sentiment label (positive or negative) based on TextBlob’s sentiment polarity.
    
(5.2) Split the dataset into training and testing sets.
    
(5.3) Convert the text data into numeric data by using TF-IDF.

(5.4) Train a Logistic Regression model and evaluate its performance on the test set. Print the classification report.

In [65]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


# (5.1) Creating a binary sentiment label based on TextBlob's sentiment polarity
df['sentiment_label'] = df['text'].apply(lambda x: 'positive' if TextBlob(x).sentiment.polarity > 0 else 'negative')
    
# (5.2)
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['sentiment_label'], test_size=0.25, random_state=0)

#(5.3)
num = TfidfVectorizer()
X_train_tfidf = num.fit_transform(X_train)
X_test_tfidf = num.transform(X_test)
# Explanation for why we only transform 2nd time:
'''
    The TF-IDF vectorizer must only be trained on X_train using .fit_transform().  
    Then, for X_test, we apply the same transformation using .transform().  
    Using .fit_transform(X_test) instead would relearn the vocabulary on test data, causing data leakage and leading to incorrect model evaluation.
'''

lrm = LogisticRegression(max_iter=1000)  # alternate: liblinear (max_iter=1000, solver='liblinear', penalty='l1', random_state=42)


lrm.fit(X_train_tfidf, y_train)
y_predicted = lrm.predict(X_test_tfidf)

#(5.4)

print(classification_report(y_test, y_predicted))


              precision    recall  f1-score   support

    negative       0.73      0.79      0.76       258
    positive       0.76      0.69      0.72       242

    accuracy                           0.74       500
   macro avg       0.75      0.74      0.74       500
weighted avg       0.75      0.74      0.74       500



## <font color='blue'>Bonus Problem (Optional): Sentiment Analysis using SentiWordNet</font>

Use <b>SentiWordNet</b> to calculate sentiment scores for each text entry in your dataset.

- <b>Step 1:</b> Calculate sentiment scores (positive, negative, objective) for each text entry.  
- <b>Step 2:</b> Assign a <b>sentiment category</b> (Positive, Negative, Neutral) based on the scores.  
- <b>Step 3:</b> Display the first 10-15 entries to inspect the sentiment scores.

In [66]:
import nltk
from nltk.corpus import sentiwordnet as swn, wordnet
from nltk import pos_tag

# Download
#nltk.download('averaged_perceptron_tagger') # For part-of-speech tagging
#nltk.download('sentiwordnet') #For accessing SentiWordNet

def get_wordnet_pos(treebank_tag):
    """Converts part-of-speech tags from the Penn Treebank tag to a WordNet tag."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        # As default case, return NOUN
        return wordnet.NOUN

def sentiment_analysis(text):
    """Performs sentiment analysis using SentiWordNet."""
    # Preprocess the text and obtain lemmatized tokens
    mytokens = word_tokenize(data_preprocess(text)) # use word_tokenize as data_preprocess() returns the joined string

# POS tagging on the lemmatized tokens
    pos_tagging = pos_tag(mytokens)

    pos_score = 0
    neg_score = 0

    # For each word and tag in pos_tagging, get the sentiment score
    for w, t in pos_tagging:
        wn_tag = get_wordnet_pos(t)
        synsets = list(swn.senti_synsets(w, pos=wn_tag))

        # If there are no synsets, skip the word
        if not synsets:
            continue

        # Take the first synset
        synset = synsets[0]
        pos_score += synset.pos_score()
        neg_score += synset.neg_score()

    # Compute the overall score
    score = pos_score - neg_score
    return score

df['sentiment_score'] = df['text'].apply(sentiment_analysis)

# Display the first 15 entries to see the sentiment scores
df[['text', 'sentiment_score']].head(10)

,text,sentiment_score
0,i just feel really helpless and heavy hearted,0.250
1,ive enjoyed being able to slouch about relax a...,1.500
2,i gave up my internship with the dmrg and am f...,-0.125
3,i dont know i feel so lost,-0.375
4,i am a kindergarten teacher and i am thoroughl...,1.250
5,i was beginning to feel quite disheartened,-0.500
6,i would think that whomever would be lucky eno...,0.500
7,i fear that they won t ever feel that deliciou...,-0.250
8,im forever taking some time out to have a lie ...,-0.250
9,i can still lose the weight without feeling de...,-1.000
